# 📈 EDA — KPI: Projeção e Meta Mensal
## Predictfy × Locaweb — FIAP Challenge 2026

Notebook exploratório: visualiza violações mensais vs meta anual e projeção de fim de ano.

**Input:** `data/raw/LW-DATASET.xlsx`
**Módulo:** `src.models.kpi_projection`

Metas anuais:
- P2 (Alta): 36–39 violações/ano
- P3 (Média): 231–263 violações/ano


In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from src.models.kpi_projection import load_data, calcular_projecao, META_ANUAL

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

kpi = load_data()
resultado = calcular_projecao(kpi)
ano = resultado["ano"]
print(f"Ano de referência: {ano}")

## 1. Violações Mensais vs Orçamento (P2 e P3)

In [ ]:
MESES_PT = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
CORES_STATUS = {"ok": "#30d158", "atencao": "#ff9f0a", "critico": "#ff2d55"}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, prio_key, prio_label in [(axes[0], "P2", "P2 — Alta"), (axes[1], "P3", "P3 — Média")]:
    dados = resultado["por_prioridade"][prio_key]
    meses_info = dados["meses"]
    meta = dados["meta_anual"]
    total = dados["total_violacoes_ano"]
    status_anual = dados["status_anual"]

    violacoes = [m["violacoes"] for m in meses_info]
    orcamento = dados["meta_anual"]["centro"] / 12
    cores = [CORES_STATUS[m["status"]] for m in meses_info]

    bars = ax.bar(MESES_PT, violacoes, color=cores, edgecolor="white", width=0.7)
    ax.axhline(orcamento, color="#1a73e8", ls="--", lw=1.5, label=f"Orçamento mensal ({orcamento:.1f})")
    ax.axhline(meta["max"] / 12, color="#ff2d55", ls=":", lw=1.5, label=f"Teto mensal ({meta['max']/12:.1f})")

    ax.set_title(
        f"{prio_label} — Total: {total} | Meta: {meta['min']}–{meta['max']} | Status: {status_anual}",
        fontsize=11, fontweight="bold"
    )
    ax.set_ylabel("Violações")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    for bar, v in zip(bars, violacoes):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.1, str(v),
                    ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.suptitle(f"Violações OLA por Mês — {ano}", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Meta Ajustada Dinâmica (Redistribuição Mensal)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, prio_key, prio_label in [(axes[0], "P2", "P2"), (axes[1], "P3", "P3")]:
    dados = resultado["por_prioridade"][prio_key]
    meses_info = dados["meses"]
    meta_centro = dados["meta_anual"]["centro"]

    violacoes = [m["violacoes"] for m in meses_info]
    meta_ajust = [m["meta_ajustada_proximo"] for m in meses_info]

    ax.bar(MESES_PT, violacoes, color="#1a73e8", alpha=0.7, label="Violações reais", width=0.5)
    ax.plot(MESES_PT, meta_ajust, color="#ff2d55", marker="o", lw=2, label="Meta ajustada próx. mês")
    ax.axhline(meta_centro / 12, color="gray", ls="--", lw=1, label=f"Base mensal ({meta_centro/12:.1f})")

    ax.set_title(f"{prio_label} — Meta Ajustada Dinâmica", fontsize=11, fontweight="bold")
    ax.set_ylabel("Violações")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Redistribuição do Orçamento de Violações ao Longo do Ano", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Resumo de Atingimento

In [ ]:
for prio_key in ["P2", "P3"]:
    dados = resultado["por_prioridade"][prio_key]
    print(f"{'='*50}")
    print(f"  {prio_key}:")
    print(f"    Total violações {ano}: {dados['total_violacoes_ano']}")
    print(f"    Meta: {dados['meta_anual']['min']}–{dados['meta_anual']['max']} violações/ano")
    print(f"    % da meta utilizada: {dados['pct_meta_utilizado']:.1f}%")
    print(f"    Projeção fim do ano: {dados['projecao_fim_ano']:.1f}")
    print(f"    Status: {dados['status_anual']}")